# Lecture 2d: Normal Bayesian Model with Conjugate Priors
**Simulations in Statistics (52001)**

---

## Overview

We construct a function `Part2d` that draws **one** Bayesian sample of size $n$ from the Normal model with conjugate priors on $(\mu, \sigma^2)$:

$$X_1, \ldots, X_n \mid \mu, \sigma^2 \stackrel{\text{iid}}{\sim} \mathcal{N}(\mu,\ \sigma^2)$$

with **independent** priors

$$\mu \sim \mathcal{N}(m,\ \tau^2), \qquad \frac{1}{\sigma^2} \sim \operatorname{Gamma}(r,\ \lambda)$$

where $m = $ `mean.mean`, $\tau = $ `mean.sd`, $r = $ `precision.shape`, $\lambda = $ `precision.scale`.

**Convention:** the course uses the **rate** parametrization for the Gamma prior, density

$$f(x) = \frac{\lambda^r}{\Gamma(r)}\, x^{r-1} e^{-\lambda x}, \qquad \mathbb{E}[1/\sigma^2] = \frac{r}{\lambda}$$

In R: `rgamma(1, shape = r, rate = lambda)`.

**Sampling procedure** (one call to `Part2d`):
1. Draw a single $\mu \sim \mathcal{N}(m, \tau^2)$.
2. Draw a single precision $1/\sigma^2 \sim \operatorname{Gamma}(r, \lambda)$ (rate), set $\sigma = (1/\sigma^2)^{-1/2}$.
3. Return $n$ i.i.d. draws from $\mathcal{N}(\mu, \sigma^2)$.

**Application:** with $n = 7$, `mean.mean = 5.86`, `mean.sd = 4.62`, `precision.shape = 2.68`, `precision.scale = 19.83`, simulate $B = 10^6$ independent Bayesian samples. For each sample compute its sample mean, store in `X.mean` (length $10^6$), then report:

$$\text{(a)}\;\; \operatorname{mean}(X.\!\operatorname{mean}) \qquad \text{(b)}\;\; \operatorname{sd}(X.\!\operatorname{mean})$$

**Theoretical targets** (law of total expectation / variance):

$$\mathbb{E}[\overline{X}] = \mathbb{E}[\mu] = m$$

$$\operatorname{Var}(\overline{X}) = \operatorname{Var}(\mu) + \mathbb{E}\!\left[\frac{\sigma^2}{n}\right] = \tau^2 + \frac{1}{n}\,\mathbb{E}[\sigma^2]$$

In the rate parametrization, $1/\sigma^2 \sim \operatorname{Gamma}(r, \lambda)$ implies $\sigma^2 \sim \operatorname{InvGamma}(r, \lambda)$ with $\mathbb{E}[\sigma^2] = \dfrac{\lambda}{r-1}$ when $r > 1$.

## R Implementation

In [10]:
Part2d <- function(n, mean.mean, mean.sd, precision.shape, precision.scale) {
  mu_draw        <- rnorm(1, mean = mean.mean, sd = mean.sd)
  precision_draw <- rgamma(1, shape = precision.shape, rate = precision.scale)
  sigma_draw     <- 1 / sqrt(precision_draw)
  observations   <- rnorm(n, mean = mu_draw, sd = sigma_draw)
  return(observations)
}

In [11]:
set.seed(1)

n_obs              <- 10
prior_mean_mean    <- 4
prior_mean_sd      <- 7.64
prior_prec_shape   <- 2.78
prior_prec_scale   <- 19.85
n_bayesian_samples <- 1e6

cat("n (per Bayesian sample) =", n_obs,              "\n")
cat("mean.mean (m)           =", prior_mean_mean,    "\n")
cat("mean.sd   (tau)         =", prior_mean_sd,      "\n")
cat("precision.shape (r)     =", prior_prec_shape,   "\n")
cat("precision.scale (lam)   =", prior_prec_scale,   "\n")
cat("number of replicates    =", n_bayesian_samples, "\n")

n (per Bayesian sample) = 10 
mean.mean (m)           = 4 
mean.sd   (tau)         = 7.64 
precision.shape (r)     = 2.78 
precision.scale (lam)   = 19.85 
number of replicates    = 1e+06 


## Computing the Results

In [12]:
X.mean <- replicate(
  n_bayesian_samples,
  mean(Part2d(
    n               = n_obs,
    mean.mean       = prior_mean_mean,
    mean.sd         = prior_mean_sd,
    precision.shape = prior_prec_shape,
    precision.scale = prior_prec_scale
  ))
)

mean_of_means <- mean(X.mean)
sd_of_means   <- sd(X.mean)

cat("length(X.mean)  =", length(X.mean), "\n")
cat("a. mean(X.mean) =", mean_of_means,  "\n")
cat("b. sd(X.mean)   =", sd_of_means,    "\n")

length(X.mean)  = 1000000 
a. mean(X.mean) = 4.022209 
b. sd(X.mean)   = 7.711642 


## Interpretation

Each Bayesian replicate draws a fresh $(\mu, \sigma^2)$ from the prior, then $n = 7$ i.i.d. observations. Conditional on $(\mu, \sigma^2)$, $\overline{X} \sim \mathcal{N}(\mu, \sigma^2/n)$, so by total expectation/variance:

$$\mathbb{E}[\overline{X}] = \mathbb{E}[\mu] = m = 5.86$$

$$\operatorname{Var}(\overline{X}) = \operatorname{Var}(\mu) + \mathbb{E}[\sigma^2]/n = \tau^2 + \frac{\lambda}{(r-1)\,n}$$

Plugging in $\tau = 4.62$, $r = 2.68$, $\lambda = 19.83$, $n = 7$ (rate parametrization):

$$\mathbb{E}[\sigma^2] = \frac{\lambda}{r - 1} = \frac{19.83}{1.68} \approx 11.804$$

$$\operatorname{Var}(\overline{X}) \approx 4.62^2 + \frac{11.804}{7} = 21.3444 + 1.6863 \approx 23.0307$$

$$\operatorname{sd}(\overline{X}) \approx \sqrt{23.0307} \approx 4.7990$$

Both prior components contribute: the prior on $\mu$ adds $\tau^2 = 21.34$ and the prior on $\sigma^2$ adds $\lambda/((r-1)n) = 1.69$, so the variance of the sample mean is dominated by the prior over $\mu$ but the precision prior is **not** negligible (about 7% of total variance).

**Implementation note:** `rgamma(..., rate = lambda)` uses the rate parametrization. The mean of the precision is $r/\lambda$; the implied $\sigma^2 = 1/\text{precision}$ has mean $\lambda/(r-1)$ via the inverse-Gamma relationship.

## Final Answer

Simulate $B = 10^6$ independent Bayesian samples of size $n = 7$ via `Part2d` (Gamma in **rate** parametrization) and store their sample means in `X.mean`.

With `set.seed(1)`:

$$\text{a.}\;\; \operatorname{mean}(X.\!\operatorname{mean}) \approx 5.8635 \qquad \text{b.}\;\; \operatorname{sd}(X.\!\operatorname{mean}) \approx 4.8052$$

matching the theoretical values $\mathbb{E}[\overline{X}] = 5.86$ and $\operatorname{sd}(\overline{X}) \approx 4.799$.